In [51]:
# Convert label spam -> phishing in full_dataset.labels.jsonl
import json
from pathlib import Path

INPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/full_dataset.labels.jsonl")
OUTPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/full_dataset.labels.phishing.jsonl")

count = 0
with INPUT_JSONL.open("r", encoding="utf-8") as f_in, OUTPUT_JSONL.open("w", encoding="utf-8") as f_out:
    for line in f_in:
        if not line.strip():
            continue
        obj = json.loads(line)
        label = str(obj.get("label", "")).lower()
        if label == "spam":
            obj["label"] = "phishing"
        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        count += 1

print(f"Wrote {count} records to {OUTPUT_JSONL}")

Wrote 13477 records to /home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/full_dataset.labels.phishing.jsonl


In [53]:
# Build combined dataset with target class counts
import pandas as pd
from pathlib import Path

SEED = 42

CEAS_FILE = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/CEAS_08.labeled.jsonl")
FULL_FILE = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/full_dataset.labels.phishing.jsonl")
NAZARIO_FILE = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/Nazario.labeled.jsonl")

OUT_FILE = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final_dataset.jsonl")

# Targets
NAZARIO_PHISH = 300
FULL_PHISH = 1700
FULL_VALID = 2000
CEAS_VALID = 1000
CEAS_SPAM = 3000

def load_jsonl(path: Path) -> pd.DataFrame:
    return pd.read_json(path, lines=True)

def sample_exact(df: pd.DataFrame, n: int, label: str, source: str) -> pd.DataFrame:
    if len(df) < n:
        raise RuntimeError(f"Not enough {label} samples in {source}: need {n}, have {len(df)}")
    return df.sample(n=n, random_state=SEED)

# Load
ceas = load_jsonl(CEAS_FILE)
full = load_jsonl(FULL_FILE)
nazario = load_jsonl(NAZARIO_FILE)

# Normalize labels to lowercase for filtering
for df in (ceas, full, nazario):
    if "label" in df.columns:
        df["label"] = df["label"].astype(str).str.lower()

# Phishing samples
nazario_phish = sample_exact(nazario[nazario["label"] == "phishing"], NAZARIO_PHISH, "phishing", "Nazario")
full_phish = sample_exact(full[full["label"] == "phishing"], FULL_PHISH, "phishing", "full_dataset")

# Valid samples
full_valid = sample_exact(full[full["label"] == "valid"], FULL_VALID, "valid", "full_dataset")
ceas_valid = sample_exact(ceas[ceas["label"] == "valid"], CEAS_VALID, "valid", "CEAS_08")

# Spam samples
ceas_spam = sample_exact(ceas[ceas["label"] == "spam"], CEAS_SPAM, "spam", "CEAS_08")

# Combine and shuffle
final_df = pd.concat(
    [nazario_phish, full_phish, full_valid, ceas_valid, ceas_spam],
    ignore_index=True,
)
final_df = final_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Save
final_df.to_json(OUT_FILE, orient="records", lines=True, force_ascii=False)
print(f"Wrote {len(final_df)} records to {OUT_FILE}")

Wrote 8000 records to /home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final_dataset.jsonl
